# 🧬 **Projeto SIEST: Sistema de Inteligência Epidemiológica e Socioespacial**
## Notebook 01: Extração e Transformação (Camada Bronze)

Este notebook é o motor de aquisição de dados do projeto. Ele é responsável por extrair os dados brutos de morbidade do SINAN (Sistema de Informação de Agravos de Notificação) através do FTP do DataSUS.

### 0. Configuração do Ambiente (Environment Setup)
Instalação das bibliotecas governamentais necessárias para a extração (PySUS).

In [ ]:
# Instala as bibliotecas de forma a garantir a compatibilidade
!pip install numpy>=2.0.0 pandas pyarrow pysus --upgrade -q

Traceback (most recent call last):
  File "/usr/local/bin/pip3", line 10, in <module>
    sys.exit(main())
             ^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/pip/_internal/cli/main.py", line 78, in main
    command = create_command(cmd_name, isolated=("--isolated" in cmd_args))
              ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/pip/_internal/commands/__init__.py", line 114, in create_command
    module = importlib.import_module(module_path)
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/lib/python3.12/importlib/__init__.py", line 90, in import_module
    return _bootstrap._gcd_import(name[level:], package, level)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "<frozen importlib._bootstrap>", line 1387, in _gcd_import
  File "<frozen importlib._bootstrap>", line 1360, in _find_and_load
  File "<frozen importlib._bootstrap>", line 1331, in _find_and_load_unloc

KeyboardInterrupt: 

### 1. Importação de Bibliotecas e Configurações Globais

In [6]:
import pandas as pd
import os
import asyncio
import httpx
from pysus.api.client import PySUS

print("Importação realizada com sucesso!")

Importação realizada com sucesso!


In [7]:
# Forçamos a biblioteca httpx (usada pelo PySUS) a ter 120 segundos de
# tolerância em vez dos 5 segundos padrão. Isso evita o ReadTimeout
if not hasattr(httpx, "_timeout_patched"):
    original_init = httpx.AsyncClient.__init__

    def nova_init(self, *args, **kwargs):
        kwargs['timeout'] = httpx.Timeout(120.0) # 2 minutos de paciência
        original_init(self, *args, **kwargs)

    httpx.AsyncClient.__init__ = nova_init
    httpx._timeout_patched = True # Trava ativada

print(f"✅ Configurações prontas! Tolerância de rede ampliada para 120s.")

✅ Configurações prontas! Tolerância de rede ampliada para 120s.


In [8]:
# Configurações Globais do Pipeline
IBGE_CAMPINAS = 350950
anos_analise = range(2014, 2027)
doencas = ['DENG', 'CHIK', 'ZIKA', 'LEPT', 'HEPA']

# Dicionário de colunas de interesse para o Grafo
colunas_alvo = [
    'ID_MUNICIP', 'ID_UNIDADE', 'DT_NOTIFIC', 'DT_SIN_PRI',
    'NU_IDADE_N', 'CS_SEXO', 'HOSPITALIZ', 'CLASSI_FIN', 'EVOLUCAO', 'CRITERIO'
]

PASTA_BRONZE = 'dados_sinan_parquet'
os.makedirs(PASTA_BRONZE, exist_ok=True)

### 2. Processamento

In [9]:
async def extrair_dados_sinan():
    print("🚀 A iniciar Motor de Extração Assíncrono (PySUS 2.0 + DuckLake)...")

    async with PySUS() as pysus:
        for doenca in doencas:
            print(f"\n--- 🧬 A processar: {doenca} ---")

            for ano in anos_analise:
                try:
                    # 1. Query Nacional
                    ficheiros_nuvem = await pysus.query(
                        dataset="sinan",
                        group=doenca,
                        year=ano
                    )

                    if not ficheiros_nuvem:
                        print(f"[{ano}] Sem registos no servidor para {doenca}.")
                        continue

                    lista_campinas = []

                    # 2. Download e Filtro
                    for ficheiro in ficheiros_nuvem:
                        local_file = await pysus.download(ficheiro)
                        df_chunk = pd.read_parquet(local_file.path)

                        # Filtra apenas o município de Campinas (Garante que só SP/Campinas passa!)
                        if 'ID_MUNICIP' in df_chunk.columns:
                            mascara = df_chunk['ID_MUNICIP'].astype(str).str.startswith(str(IBGE_CAMPINAS), na=False)
                            df_filtrado = df_chunk[mascara].copy()

                            if not df_filtrado.empty:
                                colunas_existentes = df_filtrado.columns.intersection(colunas_alvo)
                                lista_campinas.append(df_filtrado[colunas_existentes])

                        del df_chunk
                        del df_filtrado

                    # 3. Consolidação e Escrita na Camada Bronze
                    if lista_campinas:
                        df_campinas_final = pd.concat(lista_campinas, ignore_index=True)
                        nome_ficheiro = f"{PASTA_BRONZE}/{doenca}_campinas_{ano}.parquet"

                        df_campinas_final.to_parquet(nome_ficheiro, index=False, compression='snappy')
                        print(f"[{ano}] Sucesso! {len(df_campinas_final)} casos guardados.")
                        del df_campinas_final
                    else:
                        print(f"[{ano}] 0 casos isolados em Campinas.")

                except Exception as e:
                    print(f"[{ano}] ❌ Erro ao processar o pacote: {e}")

# Inicia a execução do Pipeline
await extrair_dados_sinan()

print("\n✅ ETL da Camada Bronze concluído com sucesso!")

🚀 A iniciar Motor de Extração Assíncrono (PySUS 2.0 + DuckLake)...

--- 🧬 A processar: DENG ---
[2014] Sucesso! 47646 casos guardados.
[2015] Sucesso! 77190 casos guardados.
[2016] Sucesso! 10525 casos guardados.
[2017] Sucesso! 4483 casos guardados.
[2018] Sucesso! 2880 casos guardados.
[2019] Sucesso! 32170 casos guardados.
[2020] Sucesso! 4344 casos guardados.
[2021] Sucesso! 8071 casos guardados.
[2022] Sucesso! 12446 casos guardados.
[2023] Sucesso! 12078 casos guardados.
[2024] Sucesso! 128616 casos guardados.
[2025] Sucesso! 49447 casos guardados.
[2026] Sucesso! 4033 casos guardados.

--- 🧬 A processar: CHIK ---
[2014] Sem registos no servidor para CHIK.
[2015] Sucesso! 40 casos guardados.
[2016] Sucesso! 71 casos guardados.
[2017] Sucesso! 195 casos guardados.
[2018] Sucesso! 220 casos guardados.
[2019] Sucesso! 251 casos guardados.
[2020] Sucesso! 65 casos guardados.
[2021] Sucesso! 75 casos guardados.
[2022] Sucesso! 138 casos guardados.
[2023] Sucesso! 161 casos guardados.
